In [166]:
import pandas as pd

In [167]:
!pip install xlrd

In [168]:
shark_attack = pd.read_excel("./GSAF5.xls")

In [169]:
#show columns

shark_attack.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species ',
       'Source', 'pdf', 'href formula', 'href', 'Case Number', 'Case Number.1',
       'original order', 'Unnamed: 21', 'Unnamed: 22'],
      dtype='str')

In [170]:
#drop the columns we don't need

shark_attack = shark_attack.drop(
    columns=[
        'Source', 'pdf', 'href formula', 'href',
        'Case Number', 'Case Number.1', 'original order',
        'Unnamed: 21', 'Unnamed: 22'
    ]
)

In [171]:
#drop the rows older than 1975 (left 50 years)
shark_attack = shark_attack[shark_attack['Year'] > 1975]

In [77]:
shark_attack

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species
0,18th September,2026.0,Unprovoked,Australia,Western Australia,Sorrento Beach Perth,Swimming,Greg O'Neil,M,63,Body not recovered,Y,1015hrs,Great White Shark 6m (20ft)
1,16th September,2026.0,Unprovoked,Canada,Quebec,Off the coast of Perce Le Bilbo dive site,Diving,Unknown Male,M,?,Injuries to chest shoulder and back,N,1030hrs,Great White Shark
2,14th September,2026.0,Unprovoked,Bahamas,Bimini,Bimini Island,Swimming,Unknown Austrian Tourist,F,37,Serious injuries to right ankle,N,1730hrs,Unknown
3,13th September,2026.0,Unprovoked,Australia,Western Australia,Geraldton,Surfing,Mel Ismail,M,50's,Foot lost in attack,N,0945hrs,Unknown
4,7th September,2026.0,Unprovoked,USA,Hawaii,Honolulu,Surfing,Wants to remain anonymous,M,24,Scrapes on inside of legs,N,1640hrs,Tiger Shark suspected
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,12-Jan-1976,1976.0,Unprovoked,AUSTRALIA,Queensland,Harvey Bay,NaN,Marlene Evler,F,NaN,Survived,N,NaN,NaN
3996,11-Jan-1976,1976.0,Provoked,SOUTH AFRICA,Western Cape Province,Kalk Bay,Fishing for snoek & yellowtail,"7 m fishing boat Metoo, occupants: Nicky & Pau...",NaN,NaN,"Hooked shark leapt onboard & into fish well, w...",N,07h30,"White shark, 3 m [10']"
3997,08-Jan-1976,1976.0,Unprovoked,USA,Florida,"Off Fort Pierce, St Lucie County",Spearfishing / scuba diving,Hank Greenberg,M,25,Puncture wounds to head & neck,N,NaN,6' shark
3998,02-Jan-1976,1976.0,Unprovoked,NEW ZEALAND,North Island,"Te Kaha, East coast",Spearfishing,John Grainger Leith,M,NaN,FATAL,Y,13h00,Bronze whaler shark


# Cleaning data values for Fatal, Species and Time columns

## Fatal Y/N

In [172]:
# checking all entries for a column Fatal
shark_attack['Fatal Y/N'].value_counts()

Fatal Y/N
N          3258
Y           438
UNKNOWN      24
F             5
M             3
n             1
Nq            1
2017          1
Y x 2         1
Name: count, dtype: int64

In [173]:
# cleaning values for Fatal Y/N to have only: Y, N, UNKNOWN
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].replace(["F", "M", "Y x 2"], "Y")

In [174]:
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].replace(["n", "Nq"], "N")

In [175]:
# cleaning values for Time, first checking how many unique values we have
shark_attack['Time'].nunique()

434

In [176]:
# Extract and replace digits using RegEx and converts to a string. \D+ for all strings that do not contain digits or characters

cleaned = shark_attack['Time'].astype(str).str.replace(r'\D+', '', regex=True)

In [177]:
# See the distribution of digit counts
print(cleaned.str.len().value_counts())

Time
4.0     2228
0.0      437
8.0       16
3.0        5
6.0        2
1.0        2
2.0        2
5.0        2
10.0       1
Name: count, dtype: int64


In [178]:
#removing values with 8 digits, after a first check, they could be replaced to first 4 digits
cleaned = cleaned.mask(cleaned.str.len() == 8, cleaned.str[:4])

In [179]:
#check again the distribution, the 3-10 digits due to low number can be ignored
print(cleaned.str.len().value_counts())

Time
4.0     2244
0.0      437
3.0        5
6.0        2
1.0        2
2.0        2
5.0        2
10.0       1
Name: count, dtype: int64


In [180]:
# Keeping only valid 4-digit strings, convert everything else to NaN, and format to HH:MM
shark_attack['Time'] = pd.to_datetime(
    cleaned.where(cleaned.str.len() == 4),
    format='%H%M',
    errors='coerce'
).dt.strftime('%H:%M')

In [181]:
# Checking the column entries
shark_attack['Time'].head(20)

0     10:15
1     10:30
2     17:30
3     09:45
4     16:40
5     11:00
6     16:35
7     13:40
8     13:30
9     17:00
10      NaN
11    16:15
12    13:00
13    17:15
14    17:30
15      NaN
16    17:00
17      NaN
18    09:00
19      NaN
Name: Time, dtype: str

## Species

In [182]:
shark_attack.columns

Index(['Date', 'Year', 'Type', 'Country', 'State', 'Location', 'Activity',
       'Name', 'Sex', 'Age', 'Injury', 'Fatal Y/N', 'Time', 'Species '],
      dtype='str')

In [183]:
#fixing Species column name by removing the space
shark_attack.rename(columns = {'Species ': 'Species'}, inplace = True)

In [184]:
# checking most mentioned shark types
shark_attack['Species'].value_counts().head(30)

Species
White shark                                           137
Shark involvement not confirmed                        73
Bull shark                                             62
Tiger shark                                            62
Shark involvement prior to death was not confirmed     52
Invalid                                                42
4' shark                                               37
6' shark                                               28
4' to 5' shark                                         25
Blacktip shark                                         22
5' shark                                               22
Unknown                                                21
3' shark                                               21
2 m shark                                              21
No shark involvement                                   21
3' to 4' shark                                         20
Wobbegong shark                                        20
1.2 m 

In [185]:
import numpy as np

# Define conditions for the most mentioned shark types
conditions = [
    shark_attack['Species'].str.contains('white', case=False, na=False),
    shark_attack['Species'].str.contains('tiger', case=False, na=False),
    shark_attack['Species'].str.contains('bull', case=False, na=False),
    shark_attack['Species'].str.contains('nurse', case=False, na=False),
    shark_attack['Species'].str.contains('blacktip', case=False, na=False),
    shark_attack['Species'].str.contains('wobbegong', case=False, na=False),
    shark_attack['Species'].str.contains('raggedtooth', case=False, na=False),
    shark_attack['Species'].str.contains('lemon', case=False, na=False),
    shark_attack['Species'].str.contains('bronze whaler', case=False, na=False),
]

# Define corresponding standard names
choices = [
    'White Shark',
    'Tiger Shark',
    'Bull Shark',
    'Nurse Shark',
    'Blacktip Shark',
    'Wobbegong Shark',
    'Raggedtooth Shark',
    'Lemon Shark',
    'Bronze Whaler Shark',
]

# Assign to an existing column (anything else becomes 'Other/Unknown'), with that we remove the null values as well
shark_attack['Species'] = np.select(conditions, choices, default='Other/Unknown')

# Cleaning Null Values for Fatal and Time columns

In [186]:
# Count the number of null values in both columns
shark_attack[["Fatal Y/N", "Time"]].isna().sum()

Fatal Y/N     266
Time         1754
dtype: int64

In [187]:
# renaming all Null values
shark_attack["Time"] = shark_attack["Time"].fillna("No time specified")

In [188]:
shark_attack["Fatal Y/N"] = shark_attack["Fatal Y/N"].fillna("UNKNOWN")

In [189]:
shark_attack["Fatal Y/N"].value_counts()

Fatal Y/N
N          3260
Y           447
UNKNOWN     290
2017          1
Name: count, dtype: int64

In [190]:
# for some reason there was "2017" that could not be located by using ==, therefore, it was be removed by using .contains()
shark_attack['Fatal Y/N'] = shark_attack['Fatal Y/N'].astype(str).str.contains('2017', na=False)

In [194]:
#final check if we missed any null values
shark_attack[["Time","Species","Fatal Y/N"]].isnull().sum()

Time         0
Species      0
Fatal Y/N    0
dtype: int64